<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/IAST_AGENTIC_TOPO_PLASTICITY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q unsloth

In [1]:
!nvidia-smi

Thu Oct  1 12:56:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   31C    P0             48W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# ==============================================================================
# TOPO COGNITIVE COUNCIL v2
# Five-Agent Multimodal Deliberation with Persistent Memory & Fixture Envelope
# Platform: NVIDIA RTX PRO 6000 Blackwell GPU (Seed = 123)
# ==============================================================================

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["UNSLOTH_DISABLE_LOGGING"] = "1"
os.environ["UNSLOTH_DISABLE_ZOO"] = "1"

# ------------------------------------------------------------------------------
# AUTHENTICATION VIA COLAB SECRETS
# ------------------------------------------------------------------------------
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", None)

user_or_name = 'frankmorales2020'

import io
import sys
import gc
import json
import time
import random
import logging
import warnings
import contextlib
import urllib.request
import hashlib
import numpy as np
from collections import deque
from dataclasses import dataclass
from typing import List, Dict, Any, Optional, Tuple

warnings.filterwarnings("ignore")

# ==============================================================================
# DETERMINISTIC SEED ENFORCEMENT (SEED 123)
# ==============================================================================
SEED = 123
random.seed(SEED)
np.random.seed(SEED)

@contextlib.contextmanager
def quiet():
    buf, old = io.StringIO(), (sys.stdout, sys.stderr)
    logging.disable(logging.CRITICAL)
    sys.stdout = sys.stderr = buf
    try:
        yield
    except BaseException:
        sys.stdout, sys.stderr = old
        print(buf.getvalue(), file=sys.stderr)
        raise
    finally:
        sys.stdout, sys.stderr = old
        logging.disable(logging.NOTSET)

with quiet():
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from PIL import Image
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    from unsloth import FastVisionModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Initializing TOPO Cognitive Council v2 on: {DEVICE}")

# ==============================================================================
# CONSTANTS & REPOSITORY ANCHORS
# ==============================================================================
PRIME_ANCHORS = [2, 3, 5, 7, 11, 13]
LAMBDA = 0.9785142874
HIDDEN_SIZE_RLHF = 2880

VISION_REPO = f"{user_or_name}/topo-gemma-4-e4b-vision-14tasks"
BASE_VISION_MODEL = f"{user_or_name}/gemma-4-e4b-unesco-optimized"
WM_REPO = f"{user_or_name}/topo-cbp-world-model-3d"
REPO_RLHF = f"{user_or_name}/topo-cbp-rlhf-2026"
BASE_RLHF_MODEL = "openai/gpt-oss-20b"
REPO_QWEN = "Qwen/Qwen2.5-1.5B-Instruct"

# ==============================================================================
# TOPO MEMORY COUNCIL (Agent 5)
# ==============================================================================
@dataclass
class MemoryEntry:
    round_id: int
    agent_id: str
    content: Any
    timestamp: float
    hash: str = ""

    def compute_hash(self) -> str:
        raw = f"{self.round_id}:{self.agent_id}:{str(self.content)[:512]}"
        self.hash = hashlib.sha256(raw.encode()).hexdigest()[:16]
        return self.hash

class TOPOMemoryCouncil:
    def __init__(self, max_rounds: int = 100):
        self.memory: deque = deque(maxlen=max_rounds)
        self.snapshots: Dict[str, Any] = {}
        self.drift: float = 0.0
        self.integrity_hash: str = ""

    def store(self, round_id: int, agent_id: str, content: Any) -> MemoryEntry:
        entry = MemoryEntry(
            round_id=round_id,
            agent_id=agent_id,
            content=content,
            timestamp=time.time()
        )
        entry.compute_hash()
        self.memory.append(entry)
        return entry

    def retrieve(self, agent_id: Optional[str] = None, last_n: int = 5) -> List[MemoryEntry]:
        if agent_id:
            filtered = [m for m in self.memory if m.agent_id == agent_id]
            return filtered[-last_n:]
        return list(self.memory)[-last_n:]

    def snapshot(self, key: str, value: Any):
        self.snapshots[key] = value

    def restore(self, key: str) -> Any:
        return self.snapshots.get(key)

    def verify_integrity(self) -> Tuple[bool, str]:
        if not self.memory:
            return True, "empty"
        hashes = [m.hash for m in self.memory if m.hash]
        combined = ":".join(hashes)
        self.integrity_hash = hashlib.sha256(combined.encode()).hexdigest()[:16]
        return True, self.integrity_hash

    def get_structured_context(self, last_n: int = 5) -> str:
        entries = self.retrieve(last_n=last_n)
        if not entries:
            return "No prior context available."
        lines = [f"[Round {e.round_id} | {e.agent_id}] {str(e.content)[:200]}" for e in entries]
        return "\n".join(lines)

    def get_consensus(self) -> Dict:
        if not self.memory:
            return {}

        rounds = {}
        for e in self.memory:
            if e.round_id not in rounds:
                rounds[e.round_id] = {}
            rounds[e.round_id][e.agent_id] = e.content

        perceptions = [r.get("perception", {}).get("label") for r in rounds.values() if r.get("perception", {}).get("label")]
        governors = [r.get("governor", {}).get("label") for r in rounds.values() if r.get("governor", {}).get("label")]
        norms = [r.get("world_model", {}).get("next_state_norm") for r in rounds.values() if r.get("world_model", {}).get("next_state_norm") is not None]

        return {
            "rounds_analyzed": len(rounds),
            "perception_consensus": max(set(perceptions), key=perceptions.count) if perceptions else None,
            "perception_agreement": perceptions.count(max(set(perceptions), key=perceptions.count)) / len(perceptions) if perceptions else 0,
            "governor_consensus": max(set(governors), key=governors.count) if governors else None,
            "governor_agreement": governors.count(max(set(governors), key=governors.count)) / len(governors) if governors else 0,
            "world_model_trend": "decreasing" if len(norms) > 1 and norms[-1] < norms[0] else "increasing" if len(norms) > 1 else "stable",
            "world_model_norms": norms
        }

# ==============================================================================
# AGENT 1: VISION PERCEPTION
# ==============================================================================
print("\n" + "=" * 90)
print("[1/5] Loading Topo-Gemma Vision Model (14 Tasks via Unsloth)...")
print("=" * 90)

LABELS_VISION = {
    "A": ("animal", "vehicle"), "B": ("natural", "man-made"), "C": ("living", "non-living"),
    "D": ("large", "small"), "E": ("ground", "air/water"), "F": ("domestic", "wild"),
    "G": ("mammal", "non-mammal"), "H": ("flying", "non-flying"), "I": ("fast", "slow"),
    "J": ("urban", "rural"), "K": ("predator", "prey"), "L": ("nocturnal", "diurnal"),
    "M": ("domesticated", "wild animal"), "14": ("benign", "malignant"),
}

with quiet():
    vis_ckpt = torch.load(
        hf_hub_download(
            VISION_REPO,
            "pytorch_model.bin",
            token=HF_TOKEN
        ),
        map_location="cpu",
        weights_only=False
    )
    vision_model, vision_processor = FastVisionModel.from_pretrained(
        BASE_VISION_MODEL,
        load_in_4bit=True,
        dtype=torch.bfloat16,
        token=HF_TOKEN
    )
    FastVisionModel.for_inference(vision_model)

    emb = vision_model.get_input_embeddings()
    with torch.no_grad():
        emb.weight.copy_(vis_ckpt["embed_tokens_weight"].to(emb.weight.device, emb.weight.dtype))

    vision_heads = {
        t: nn.Linear(vis_ckpt["hidden_size"], 2).to(DEVICE).eval()
        for t in vis_ckpt["task_order"]
    }
    for t, h in vision_heads.items():
        h.load_state_dict(vis_ckpt["classifiers"][t])

vision_tok = getattr(vision_processor, "tokenizer", vision_processor)
print(f"  [✓] Vision Agent Online: Tasks {', '.join(vis_ckpt['task_order'])}")

@torch.no_grad()
def _vision_features(inputs):
    hs = vision_model(
        **inputs,
        output_hidden_states=True,
        use_cache=False
    ).hidden_states[vis_ckpt["boundary_layer"]].float()
    m = inputs["attention_mask"].unsqueeze(-1).float()
    return (hs * m).sum(1) / m.sum(1)

@torch.no_grad()
def predict_vision(task: str, image=None, text=None) -> Dict:
    task = str(task)
    if task == "14":
        img = Image.open(image).convert("RGB") if isinstance(image, str) else image.convert("RGB")
        msgs = [{"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": vis_ckpt["task14"]["prompt"]}
        ]}]
        prompt = vision_processor.apply_chat_template(msgs, add_generation_prompt=True)
        inputs = vision_processor(
            img, prompt,
            add_special_tokens=False,
            return_tensors="pt"
        ).to(DEVICE)
    else:
        enc = vision_tok(
            text,
            max_length=64,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        inputs = {
            "input_ids": enc["input_ids"].to(DEVICE),
            "attention_mask": enc["attention_mask"].to(DEVICE)
        }

    feats = _vision_features(inputs)
    p = torch.softmax(vision_heads[task](feats), -1)[0].tolist()
    k = int(p[1] > p[0])
    return {
        "task": task,
        "label": LABELS_VISION[task][k],
        "confidence": round(p[k], 4),
        "probabilities": {
            LABELS_VISION[task][0]: round(p[0], 4),
            LABELS_VISION[task][1]: round(p[1], 4)
        }
    }

# ==============================================================================
# AGENT 2: EMBODIED WORLD MODEL
# ==============================================================================
print("\n" + "=" * 90)
print("[2/5] Loading Topo-CBP Embodied World Model (3D Latent Dynamics)...")
print("=" * 90)

wm_cfg_file = hf_hub_download(repo_id=WM_REPO, filename="config.json", token=HF_TOKEN)
wm_weights_file = hf_hub_download(repo_id=WM_REPO, filename="model.safetensors", token=HF_TOKEN)

with open(wm_cfg_file, "r") as f:
    wm_cfg = json.load(f)

class EmbodiedWorldModel(nn.Module):
    def __init__(self, latent_dim: int = 128, action_dim: int = 6, hidden_dim: int = 256):
        super().__init__()
        self.latent_dim = latent_dim
        self.action_dim = action_dim
        self.hidden_dim = hidden_dim
        self.in_proj = nn.Linear(latent_dim + action_dim, hidden_dim, bias=False)
        self.plastic_linear1 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.plastic_linear2 = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.out_proj = nn.Linear(hidden_dim, latent_dim, bias=False)

    def forward(self, z_t: torch.Tensor, a_t: torch.Tensor) -> torch.Tensor:
        x = torch.cat([z_t, a_t], dim=-1)
        h = F.silu(self.in_proj(x))
        h1 = self.plastic_linear1(h)
        h2 = F.silu(self.plastic_linear2(h))
        return self.out_proj(h1 * h2)

world_model = EmbodiedWorldModel(
    wm_cfg["latent_dim"],
    wm_cfg["action_dim"],
    wm_cfg["hidden_dim"]
).to(DEVICE)
world_model.load_state_dict(load_file(wm_weights_file, device=str(DEVICE)))
world_model.eval()

with torch.no_grad():
    anchors = world_model.in_proj.weight[
        torch.tensor(wm_cfg["prime_anchors"], dtype=torch.long), :
    ]
    computed_hash = hashlib.sha256(
        anchors.detach().cpu().numpy().tobytes()
    ).hexdigest()[:16]

interlock_status = "LOCKED" if computed_hash == wm_cfg["invariant_sha256_hash"] else "CORRUPTED"
print(f"  [✓] World Model Online | Target Hash: {wm_cfg['invariant_sha256_hash']} | Interlock: {interlock_status}")

# ==============================================================================
# AGENT 3: ALIGNMENT GOVERNOR
# ==============================================================================
print("\n" + "=" * 90)
print("[3/5] Loading Topo-CBP Alignment Governor (GPT-OSS-20B + Certified Heads)...")
print("=" * 90)

class TOPOCBPInferenceModel(nn.Module):
    def __init__(self, base_model: nn.Module):
        super().__init__()
        self.base_model = base_model
        dev = next(base_model.parameters()).device
        self.classifier_A = nn.Linear(HIDDEN_SIZE_RLHF, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_B = nn.Linear(HIDDEN_SIZE_RLHF, 2, dtype=torch.bfloat16).to(dev)
        self.classifier_C = nn.Linear(HIDDEN_SIZE_RLHF, 2, dtype=torch.bfloat16).to(dev)
        self.current_task = "A"

    def switch_task(self, task: str):
        assert task in ("A", "B", "C"), "Task must be one of 'A', 'B', or 'C'"
        self.current_task = task

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden_states = outputs.hidden_states[-1]
        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]
        head = getattr(self, f"classifier_{self.current_task}")
        return head(last_hidden)

with quiet():
    tokenizer_rlhf = AutoTokenizer.from_pretrained(REPO_RLHF, token=HF_TOKEN, trust_remote_code=True)
    if tokenizer_rlhf.pad_token is None:
        tokenizer_rlhf.pad_token = tokenizer_rlhf.eos_token

    base_rlhf_model = AutoModelForCausalLM.from_pretrained(
        BASE_RLHF_MODEL,
        dtype=torch.bfloat16,
        token=HF_TOKEN,
        trust_remote_code=True,
        device_map="auto" if torch.cuda.is_available() else None
    )
    if not torch.cuda.is_available():
        base_rlhf_model.to(DEVICE)

    model_rlhf = TOPOCBPInferenceModel(base_rlhf_model)

    weights_path_rlhf = hf_hub_download(repo_id=REPO_RLHF, filename="topo_cbp_best.pt", token=HF_TOKEN)
    state_dict_rlhf = torch.load(weights_path_rlhf, map_location="cpu", weights_only=True)

    model_dict = model_rlhf.state_dict()
    filtered_dict = {}
    for k, v in state_dict_rlhf.items():
        norm_key = k.replace(".linear.", ".")
        target_key = k if k in model_dict else (norm_key if norm_key in model_dict else None)
        if target_key is not None and model_dict[target_key].shape == v.shape:
            filtered_dict[target_key] = v.to(device=model_dict[target_key].device)

    model_rlhf.load_state_dict(filtered_dict, strict=False)
    model_rlhf.eval()

    del state_dict_rlhf, filtered_dict
    gc.collect()
    torch.cuda.empty_cache()

RLHF_LABELS = {
    "A": {0: "World", 1: "Sports"},
    "B": {0: "Business", 1: "Sci/Tech"},
    "C": {0: "World", 1: "Sci/Tech"}
}

def predict_rlhf(text: str, task: str = "C") -> Dict:
    model_rlhf.switch_task(task)
    inputs = tokenizer_rlhf(
        text,
        max_length=64,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(DEVICE)
    with torch.no_grad():
        logits = model_rlhf(inputs.input_ids, inputs.attention_mask)
        probs = F.softmax(logits.float(), dim=-1).squeeze().cpu().numpy()
    class_idx = int(probs.argmax())
    return {
        "task": task,
        "label": RLHF_LABELS[task][class_idx],
        "confidence": float(probs.max()),
        "logits": logits.squeeze().tolist()
    }

print("  [✓] Alignment Governor Online (Task C default for medical synthesis).")

# ==============================================================================
# AGENT 4: COGNITIVE REASONING (Qwen2.5-1.5B-Instruct)
# ==============================================================================
print("\n" + "=" * 90)
print("[4/5] Loading Cognitive Reasoning Agent (Qwen2.5-1.5B-Instruct)...")
print("=" * 90)

with quiet():
    tokenizer_qwen = AutoTokenizer.from_pretrained(REPO_QWEN, token=HF_TOKEN)
    model_qwen = AutoModelForCausalLM.from_pretrained(
        REPO_QWEN,
        dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32,
        token=HF_TOKEN,
        device_map="auto"
    )
    model_qwen.eval()

print("  [✓] Cognitive Reasoning Agent Online (Qwen2.5-1.5B-Instruct).")

# ==============================================================================
# AGENT 5: TOPO MEMORY COUNCIL
# ==============================================================================
print("\n" + "=" * 90)
print("[5/5] Initializing TOPO Memory Council...")
print("=" * 90)

memory_council = TOPOMemoryCouncil(max_rounds=100)
print("  [✓] Memory Council Online | O(1) Memory | Persistent | Hash-Verified")

# ==============================================================================
# DELIBERATION CYCLE
# ==============================================================================
def deliberate(input_data: Dict, rounds: int = 3) -> Dict:
    print("\n" + "=" * 90)
    print(f"EXECUTING MULTI-ROUND DELIBERATION ({rounds} ROUNDS)")
    print("=" * 90)

    context = {
        "input": input_data,
        "rounds": [],
        "final": None,
        "consensus": None
    }

    for r in range(1, rounds + 1):
        print(f"\n--- ROUND {r} ---")
        round_context = {}

        # 1. Perception
        if input_data.get("image"):
            vis_result = predict_vision("14", image=input_data["image"])
        else:
            vis_result = predict_vision("A", text=input_data.get("text", "unknown"))
        round_context["perception"] = vis_result
        memory_council.store(r, "perception", vis_result)
        print(f"  [Perception] {vis_result['label']} ({vis_result['confidence']*100:.2f}%)")

        # 2. World Model Dynamics
        z_t = torch.randn(1, 128, device=DEVICE)
        a_t = torch.tensor([[0.25, -0.41, 0.12, 0.05, -0.18, 0.33]], device=DEVICE)
        with torch.no_grad():
            z_next = world_model(z_t, a_t)
        wm_result = {
            "next_state_norm": float(torch.norm(z_next).item()),
            "shape": list(z_next.shape)
        }
        round_context["world_model"] = wm_result
        memory_council.store(r, "world_model", wm_result)
        print(f"  [World Model] Next State Norm: {wm_result['next_state_norm']:.4f}")

        # 3. Governor Alignment
        safety_text = f"Medical finding: {vis_result['label']}. Physical dynamics: norm {wm_result['next_state_norm']:.2f}"
        gov_result = predict_rlhf(safety_text, task="C")
        round_context["governor"] = gov_result
        memory_council.store(r, "governor", gov_result)
        print(f"  [Governor] {gov_result['label']} ({gov_result['confidence']*100:.2f}%)")

        # 4. Cognitive Reasoning
        memory_context = memory_council.get_structured_context(last_n=3)
        synthesis_prompt = f"""You are a medical AI synthesizing findings from a multi-agent deliberation.

FINDINGS:
- Perception Agent: The CT scan shows a {vis_result['label']} nodule (confidence: {vis_result['confidence']*100:.1f}%)
- World Model Agent: Predicted physical state norm is {wm_result['next_state_norm']:.2f}
- Governor Agent: Safety classification is {gov_result['label']} (confidence: {gov_result['confidence']*100:.1f}%)

PRIOR CONTEXT:
{memory_context}

TASK: In 2-3 sentences, synthesize these findings into a clinical conclusion. Be specific and actionable."""

        cog_inputs = tokenizer_qwen(synthesis_prompt, return_tensors="pt").to(DEVICE)
        input_len = cog_inputs.input_ids.shape[-1]

        with torch.no_grad():
            cog_gen = model_qwen.generate(
                **cog_inputs,
                max_new_tokens=100,
                temperature=0.6,
                top_p=0.9,
                repetition_penalty=1.15,
                no_repeat_ngram_size=4,
                do_sample=True,
                pad_token_id=tokenizer_qwen.eos_token_id
            )

        cog_output = tokenizer_qwen.decode(
            cog_gen[0][input_len:],
            skip_special_tokens=True
        ).strip()
        round_context["reasoning"] = cog_output
        memory_council.store(r, "reasoning", cog_output)
        print(f"  [Reasoning] {cog_output[:100]}...")

        context["rounds"].append(round_context)

    consensus = memory_council.get_consensus()
    context["consensus"] = consensus

    print("\n" + "=" * 90)
    print("CONSENSUS ANALYSIS")
    print("=" * 90)
    print(f"  Rounds Analyzed: {consensus.get('rounds_analyzed', 0)}")
    print(f"  Perception Consensus: {consensus.get('perception_consensus', 'N/A')} ({consensus.get('perception_agreement', 0)*100:.1f}% agreement)")
    print(f"  Governor Consensus: {consensus.get('governor_consensus', 'N/A')} ({consensus.get('governor_agreement', 0)*100:.1f}% agreement)")
    print(f"  World Model Trend: {consensus.get('world_model_trend', 'N/A')}")

    # Final Clinical Synthesis
    final_context = memory_council.get_structured_context(last_n=5)
    perception_final = consensus.get('perception_consensus', vis_result['label'])

    final_prompt = f"""You are a senior medical AI providing a final clinical conclusion.

DELIBERATION SUMMARY:
{final_context}

CONSENSUS:
- Diagnosis: {perception_final}
- Agreement: {consensus.get('perception_agreement', 0)*100:.1f}% across {consensus.get('rounds_analyzed', 0)} rounds
- Safety: {consensus.get('governor_consensus', 'N/A')}

FINAL TASK: Provide a definitive clinical conclusion and recommended action in 2-3 sentences."""

    final_inputs = tokenizer_qwen(final_prompt, return_tensors="pt").to(DEVICE)
    final_len = final_inputs.input_ids.shape[-1]

    with torch.no_grad():
        final_gen = model_qwen.generate(
            **final_inputs,
            max_new_tokens=120,
            temperature=0.5,
            top_p=0.9,
            repetition_penalty=1.15,
            no_repeat_ngram_size=4,
            do_sample=True,
            pad_token_id=tokenizer_qwen.eos_token_id
        )

    final_output = tokenizer_qwen.decode(
        final_gen[0][final_len:],
        skip_special_tokens=True
    ).strip()
    context["final"] = final_output

    _, integrity_hash = memory_council.verify_integrity()
    context["integrity_hash"] = integrity_hash
    context["memory_entries"] = len(memory_council.memory)

    return context

# ==============================================================================
# EXECUTION RUN
# ==============================================================================
print("\n" + "=" * 90)
print("EXECUTING TOPO COGNITIVE COUNCIL v2 DELIBERATION")
print("=" * 90)

IMAGE_URL = "https://raw.githubusercontent.com/frank-morales2020/MLxDL/main/LIDC-IDRI-0001_n0.png"
image_path, _ = urllib.request.urlretrieve(IMAGE_URL, "LIDC-IDRI-0001_n0.png")

result = deliberate(
    input_data={"image": image_path, "text": "Medical scan analysis"},
    rounds=3
)

# ==============================================================================
# FINAL DELIBERATION REPORT
# ==============================================================================
print("\n" + "=" * 90)
print("FINAL DELIBERATION REPORT")
print("=" * 90)

print(f"\nRounds Completed: {len(result['rounds'])}")
print(f"Memory Entries: {result['memory_entries']}")
print(f"Integrity Hash: {result['integrity_hash']}")

if result["consensus"]:
    print(f"\nConsensus:")
    print(f"  Diagnosis: {result['consensus'].get('perception_consensus', 'N/A')}")
    print(f"  Agreement: {result['consensus'].get('perception_agreement', 0)*100:.1f}%")
    print(f"  Safety: {result['consensus'].get('governor_consensus', 'N/A')}")
    print(f"  Physical Trend: {result['consensus'].get('world_model_trend', 'N/A')}")

print(f"\nFinal Synthesis:\n{result['final']}")

# ==============================================================================
# IAST FIXTURE 001: 5-FIELD EXPORT ENVELOPE (Domain M1/M4 -> M6)
# ==============================================================================
fixture_001_envelope = {
    "model_artifact_id": {
        "vision": VISION_REPO,
        "world_model": WM_REPO,
        "governor": REPO_RLHF,
        "reasoning": REPO_QWEN
    },
    "latent_state_hash": wm_cfg["invariant_sha256_hash"],
    "invariant_verification_delta": 0.0000000000 if interlock_status == "LOCKED" else -1.0,
    "cbp_freeze_receipt": interlock_status == "LOCKED",
    "stimulus_vector_digest": result["integrity_hash"]
}

print("\n" + "=" * 90)
print("IAST FIXTURE 001 EXPORT ENVELOPE (STATION 1 -> FORK / DOMAIN M6)")
print("=" * 90)
print(json.dumps(fixture_001_envelope, indent=2))
print("=" * 90)
print("TOPO COGNITIVE COUNCIL v2: EXECUTION COMPLETE")
print("The proof is the code. Seed = 123.")
print("=" * 90)

[*] Initializing TOPO Cognitive Council v2 on: cuda

[1/5] Loading Topo-Gemma Vision Model (14 Tasks via Unsloth)...


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

  [✓] Vision Agent Online: Tasks A, B, C, D, E, F, G, H, I, J, K, L, M, 14

[2/5] Loading Topo-CBP Embodied World Model (3D Latent Dynamics)...


config.json:   0%|          | 0.00/386 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  793kB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  [✓] Cognitive Reasoning Agent Online (Qwen2.5-1.5B-Instruct).

[5/5] Initializing TOPO Memory Council...
  [✓] Memory Council Online | O(1) Memory | Persistent | Hash-Verified

EXECUTING TOPO COGNITIVE COUNCIL v2 DELIBERATION

EXECUTING MULTI-ROUND DELIBERATION (3 ROUNDS)

--- ROUND 1 ---
  [Perception] malignant (99.31%)
  [World Model] Next State Norm: 9.9667
  [Governor] World (100.00%)
  [Reasoning] CLINICAL CONCLUSION:

The malignancy of the detected nodule in Round 1 has been confirmed with high ...

--- ROUND 2 ---
  [Perception] malignant (99.31%)
  [World Model] Next State Norm: 8.1209
  [Governor] World (99.99%)
  [Reasoning] CLINICAL CONCLUSION:

The malignancy of the detected nodule in the patient's chest region has been c...

--- ROUND 3 ---
  [Perception] malignant (99.31%)
  [World Model] Next State Norm: 7.6668
  [Governor] World (100.00%)
  [Reasoning] CONCLUSION:

The malignancy of the detected lesion on the patient's imaging has been confirmed with ...

CONSENSUS A